# 12: Cube Surgery Workflow (Julia-Accelerated)

We compare:
1. a standard cube (8 vertices, 12 edges, 6 faces triangulated),
2. a cube with an internal face "removed" (topologically modified).

Then we:
- manually define the simplicial complex with 8 vertices,
- visualize triangulations and homology generators (`H0`, `H1`, `H2`),
- run pySurgery homeomorphism diagnostics,
- perform a surgery step (add/remove faces),
- verify topology changes after surgery.

## Environment Check (Optional one-time installs)

If needed in your active kernel environment:
- `pip install juliacall plotly`
- Julia packages once (in Julia REPL): `import Pkg; Pkg.add("AbstractAlgebra")`

In [ ]:
print("Using pySurgery Julia bridge warm-up (no in-notebook Pkg.add calls).")

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time
import gudhi

from pysurgery.bridge.julia_bridge import julia_engine
from pysurgery.integrations.gudhi_bridge import triangulate_surface, extract_complex_data
from pysurgery.core.complexes import ChainComplex
from pysurgery.core.homology_generators import (
    compute_homology_basis_from_simplex_tree,
    compute_optimal_h1_basis_from_simplex_tree,
)
from pysurgery.homeomorphism import analyze_homeomorphism_2d_result
from pysurgery.homeomorphism_witness import build_homeomorphism_witness

print("Julia available:", julia_engine.available)
if not julia_engine.available:
    raise RuntimeError(
        "Julia backend is not available for this notebook run. "
        f"Bridge error: {julia_engine.error}"
    )
warmup_report = julia_engine.warmup()
print("Julia warm-up completed.")
print("  completed workloads:", len(warmup_report.get("completed", [])))
if warmup_report.get("failed"):
    print("  failed workloads:", sorted(warmup_report["failed"].keys()))

## Cube Builders
Define 8 vertices for a unit cube, then triangulate each face into 2 triangles.

In [ ]:
def create_cube_simplex_tree():
    """
    Create a cube with 8 vertices, triangulated into 12 triangles (2 per face).

    Vertices:
    0: (0,0,0), 1: (1,0,0), 2: (1,1,0), 3: (0,1,0)  [bottom]
    4: (0,0,1), 5: (1,0,1), 6: (1,1,1), 7: (0,1,1)  [top]
    """
    vertices = np.array([
        [0, 0, 0],  # 0
        [1, 0, 0],  # 1
        [1, 1, 0],  # 2
        [0, 1, 0],  # 3
        [0, 0, 1],  # 4
        [1, 0, 1],  # 5
        [1, 1, 1],  # 6
        [0, 1, 1],  # 7
    ], dtype=float)

    # Define 12 triangles (2 per face of cube)
    triangles = [
        # Bottom face (z=0)
        (0, 1, 2),
        (0, 2, 3),
        # Top face (z=1)
        (4, 6, 5),
        (4, 7, 6),
        # Front face (y=0)
        (0, 5, 1),
        (0, 4, 5),
        # Back face (y=1)
        (3, 2, 6),
        (3, 6, 7),
        # Left face (x=0)
        (0, 3, 7),
        (0, 7, 4),
        # Right face (x=1)
        (1, 5, 6),
        (1, 6, 2),
    ]

    # Create simplex tree manually
    st = gudhi.SimplexTree()

    # Add vertices
    for i in range(8):
        st.insert([i])

    # Add edges (1-skeleton)
    edges_set = set()
    for tri in triangles:
        for i in range(3):
            for j in range(i+1, 3):
                edge = tuple(sorted([tri[i], tri[j]]))
                edges_set.add(edge)

    for edge in edges_set:
        st.insert(list(edge))

    # Add triangles
    for tri in triangles:
        st.insert(list(tri))

    return st, vertices


def create_cube_with_removed_face_simplex_tree():
    """
    Create a cube with one internal face removed (removed a triangle to create
    a topological change).
    """
    vertices = np.array([
        [0, 0, 0],  # 0
        [1, 0, 0],  # 1
        [1, 1, 0],  # 2
        [0, 1, 0],  # 3
        [0, 0, 1],  # 4
        [1, 0, 1],  # 5
        [1, 1, 1],  # 6
        [0, 1, 1],  # 7
    ], dtype=float)

    # Same 12 triangles but remove 2 from one face
    triangles = [
        # Bottom face (z=0) - REMOVED for surgery
        # (0, 1, 2),
        # (0, 2, 3),
        # Top face (z=1)
        (4, 6, 5),
        (4, 7, 6),
        # Front face (y=0)
        (0, 5, 1),
        (0, 4, 5),
        # Back face (y=1)
        (3, 2, 6),
        (3, 6, 7),
        # Left face (x=0)
        (0, 3, 7),
        (0, 7, 4),
        # Right face (x=1)
        (1, 5, 6),
        (1, 6, 2),
    ]

    # Create simplex tree manually
    st = gudhi.SimplexTree()

    # Add vertices
    for i in range(8):
        st.insert([i])

    # Add edges (1-skeleton)
    edges_set = set()
    for tri in triangles:
        for i in range(3):
            for j in range(i+1, 3):
                edge = tuple(sorted([tri[i], tri[j]]))
                edges_set.add(edge)

    for edge in edges_set:
        st.insert(list(edge))

    # Add triangles
    for tri in triangles:
        st.insert(list(tri))

    return st, vertices

## Build Cube Simplicial Complexes
Directly create the cube simplicial trees (no point cloud needed).

In [ ]:
print("Creating cubes with 8 vertices...")

# Standard cube (complete)
st_clean, pts_clean = create_cube_simplex_tree()
print(f"Clean cube: {st_clean.num_vertices()} vertices, {st_clean.num_simplices()} simplices")

# Cube with bottom face removed (topological modification)
st_modified, pts_modified = create_cube_with_removed_face_simplex_tree()
print(f"Modified cube: {st_modified.num_vertices()} vertices, {st_modified.num_simplices()} simplices")

## Topology Extraction Utilities

In [ ]:
def simplex_counts(st):
    skel = list(st.get_skeleton(2))
    V = st.num_vertices()
    E = sum(1 for s, _ in skel if len(s) == 2)
    F = sum(1 for s, _ in skel if len(s) == 3)
    return V, E, F, (V - E + F)

def build_surface_record_from_simplex_tree(st, points, name):
    print(f"\n{'='*10} Processing: {name} {'='*10}")

    start_record_perf = time.perf_counter()
    measured_steps = 0.0 # We will sum the explicitly timed steps here

    # 2. Data Extraction (Julia-accelerated)
    print(f"[{name}] Extracting complex data (Julia-accelerated)...")
    t_start = time.perf_counter()
    boundaries, cells, _, _ = extract_complex_data(st)
    t_step = time.perf_counter() - t_start
    measured_steps += t_step
    print(f"[{name}] Data extraction finished: {t_step:.4f}s")

    # 3. Chain Complex Setup
    print(f"[{name}] Initializing Chain Complex...")
    t_start = time.perf_counter()
    cc = ChainComplex(boundaries=boundaries, dimensions=sorted(cells.keys()), coefficient_ring="Z")
    t_step = time.perf_counter() - t_start
    measured_steps += t_step
    print(f"[{name}] Chain Complex ready: {t_step:.4f}s")

    # 4. Homology Ranks (Detailed breakdown)
    print(f"[{name}] Beginning Homology Rank calculations...")

    t_start = time.perf_counter()
    h0 = cc.homology(0)
    t_step = time.perf_counter() - t_start
    measured_steps += t_step
    print(f"  > H_0 calculated in {t_step:.4f}s")

    t_start = time.perf_counter()
    h1 = cc.homology(1)
    t_step = time.perf_counter() - t_start
    measured_steps += t_step
    print(f"  > H_1 calculated in {t_step:.4f}s")

    t_start = time.perf_counter()
    h2 = cc.homology(2)
    t_step = time.perf_counter() - t_start
    measured_steps += t_step
    print(f"  > H_2 calculated in {t_step:.4f}s")

    # 5. Basis/Generator Computations (The heaviest part)
    print(f"[{name}] Beginning generator basis computations...")

    t_start = time.perf_counter()
    g0 = compute_homology_basis_from_simplex_tree(st, dimension=0)
    t_step = time.perf_counter() - t_start
    measured_steps += t_step
    print(f"  > H_0 basis finished: {t_step:.4f}s")

    t_start = time.perf_counter()
    g1 = compute_optimal_h1_basis_from_simplex_tree(st, point_cloud=points, max_cycles=8)
    t_step = time.perf_counter() - t_start
    measured_steps += t_step
    print(f"  > H_1 optimal basis finished: {t_step:.4f}s")

    t_start = time.perf_counter()
    g2 = compute_homology_basis_from_simplex_tree(st, dimension=2, mode="valid")
    t_step = time.perf_counter() - t_start
    measured_steps += t_step
    print(f"  > H_2 basis finished: {t_step:.4f}s")

    # 6. Final counts (Adding a timer here because list comprehensions on large skeletons can be very slow)
    print(f"[{name}] Calculating simplex counts...")
    t_start = time.perf_counter()
    V, E, F, chi = simplex_counts(st)
    t_step = time.perf_counter() - t_start
    measured_steps += t_step
    print(f"  > Counts finished in {t_step:.4f}s")

    # --- Diagnostics ---
    total_perf_time = time.perf_counter() - start_record_perf
    ghost_time = total_perf_time - measured_steps

    print(f"\n[{name}] --- TIMING DIAGNOSTICS ---")
    print(f"[{name}] Sum of explicit steps:  {measured_steps:.4f}s")
    print(f"[{name}] UNACCOUNTED GHOST TIME: {ghost_time:.4f}s  <-- (This is Garbage Collector pauses or hidden Python overhead)")
    print(f"[{name}] TOTAL WALL CLOCK TIME:  {total_perf_time:.4f}s")
    print(f"{'='*34}\n")

    return {
        "name": name,
        "points": points,
        "st": st,
        "cc": cc,
        "cells": cells,
        "counts": {"V": V, "E": E, "F": F, "chi": chi},
        "homology": {"H0": h0, "H1": h1, "H2": h2},
        "generators": {"H0": g0, "H1": g1, "H2": g2},
    }

# --- Execution ---
t_global_start = time.perf_counter()

# Build clean and modified cubes
clean = build_surface_record_from_simplex_tree(st_clean, pts_clean, "clean_cube")

print("\n\n>>> RUNNING SECOND PASS TO CHECK FOR JIT WARMUP CACHING <<<\n")
handled = build_surface_record_from_simplex_tree(st_modified, pts_modified, "cube_with_removed_face")

t_global_total = time.perf_counter() - t_global_start
print(f"COMPLETE SCRIPT ELAPSED TIME: {t_global_total:.2f}s")

In [ ]:
def print_surface_summary(rec):
    print(f"\n=== {rec['name']} ===")
    c = rec["counts"]
    print(f"V={c['V']} E={c['E']} F={c['F']} Euler={c['chi']}")
    print(f"H0={rec['homology']['H0']}")
    print(f"H1={rec['homology']['H1']}")
    print(f"H2={rec['homology']['H2']}")
    print(f"H0 generators: rank={rec['generators']['H0'].rank} | {rec['generators']['H0'].message}")
    print(f"H1 generators: rank={rec['generators']['H1'].rank} | {rec['generators']['H1'].message}")
    print(f"H2 generators: rank={rec['generators']['H2'].rank} | {rec['generators']['H2'].message}")


print_surface_summary(clean)
print_surface_summary(handled)

## Plotting Helpers (mesh + vertices + edges + H0/H1/H2 overlays)

In [ ]:
def first_h0_vertices(h0_basis, max_points=8):
    verts = []
    for g in h0_basis.generators:
        for s in g.support_simplices:
            if len(s) == 1:
                verts.append(int(s[0]))
    # unique preserving order
    uniq = []
    seen = set()
    for v in verts:
        if v not in seen:
            seen.add(v)
            uniq.append(v)
    return uniq[:max_points]


def edges_from_h1_basis(h1_basis, max_generators=4):
    edge_sets = []
    for g in h1_basis.generators[:max_generators]:
        edge_sets.append([tuple(sorted(e)) for e in g.support_edges])
    return edge_sets


def faces_from_h2_basis(h2_basis, max_faces=200):
    faces = []
    for g in h2_basis.generators:
        for s in g.support_simplices:
            if len(s) == 3:
                faces.append(tuple(s))
    # dedupe and cap
    out = []
    seen = set()
    for f in faces:
        sf = tuple(sorted(f))
        if sf not in seen:
            seen.add(sf)
            out.append(sf)
        if len(out) >= max_faces:
            break
    return out


def create_surface_plot(rec):
    """Create a single fullscreen plot for a surface with full simplicial skeleton and generators."""
    pts = rec["points"]
    st = rec["st"]
    h0 = rec["generators"]["H0"]
    h1 = rec["generators"]["H1"]
    h2 = rec["generators"]["H2"]

    fig = go.Figure()

    faces = [tuple(s) for s, _ in st.get_skeleton(2) if len(s) == 3]
    edges = [tuple(s) for s, _ in st.get_skeleton(1) if len(s) == 2]
    if len(faces) == 0:
        raise ValueError(f"No faces found in simplex tree for {rec['name']}.")

    fi = [f[0] for f in faces]
    fj = [f[1] for f in faces]
    fk = [f[2] for f in faces]

    # Base 2-skeleton surface
    fig.add_trace(
        go.Mesh3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            i=fi, j=fj, k=fk,
            opacity=0.18,
            color="lightblue",
            name="Mesh Surface (2-skeleton)",
            showlegend=True,
        )
    )

    # Base 1-skeleton edges
    if edges:
        ex, ey, ez = [], [], []
        for u, v in edges:
            ex.extend([pts[u, 0], pts[v, 0], None])
            ey.extend([pts[u, 1], pts[v, 1], None])
            ez.extend([pts[u, 2], pts[v, 2], None])
        fig.add_trace(
            go.Scatter3d(
                x=ex, y=ey, z=ez,
                mode="lines",
                line=dict(width=3, color="dimgray"),
                name="Edges (1-skeleton)",
                showlegend=True,
            )
        )

    # Base 0-skeleton vertices
    fig.add_trace(
        go.Scatter3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            mode="markers",
            marker=dict(size=5, color="royalblue", symbol="circle"),
            name="Vertices (0-skeleton)",
            showlegend=True,
        )
    )

    # H0 representatives
    h0_vs = first_h0_vertices(h0)
    if h0_vs:
        fig.add_trace(
            go.Scatter3d(
                x=pts[h0_vs, 0], y=pts[h0_vs, 1], z=pts[h0_vs, 2],
                mode="markers",
                marker=dict(size=9, color="black", symbol="diamond"),
                name="H0 Representatives",
                showlegend=True,
            )
        )

    # H1 generators as colored edge loops
    colors = ["red", "orange", "purple", "green", "magenta", "cyan"]
    for gi, edge_list in enumerate(edges_from_h1_basis(h1, max_generators=4)):
        xs, ys, zs = [], [], []
        for (u, v) in edge_list:
            xs.extend([pts[u, 0], pts[v, 0], None])
            ys.extend([pts[u, 1], pts[v, 1], None])
            zs.extend([pts[u, 2], pts[v, 2], None])

        fig.add_trace(
            go.Scatter3d(
                x=xs, y=ys, z=zs,
                mode="lines",
                line=dict(width=6, color=colors[gi % len(colors)]),
                name=f"H1 Generator {gi}",
                showlegend=True,
            )
        )

    # H2 representative faces (first generator support subset)
    h2_faces = faces_from_h2_basis(h2, max_faces=120)
    if h2_faces:
        hi = [f[0] for f in h2_faces]
        hj = [f[1] for f in h2_faces]
        hk = [f[2] for f in h2_faces]
        fig.add_trace(
            go.Mesh3d(
                x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
                i=hi, j=hj, k=hk,
                opacity=0.55,
                color="gold",
                name="H2 Generator Support",
                showlegend=True,
            )
        )

    fig.update_layout(
        title=f"Simplicial Complex: {rec['name']}",
        height=800,
        width=1200,
        showlegend=True,
        legend=dict(x=0.0, y=1.0, bgcolor="rgba(255,255,255,0.8)"),
    )
    return fig


In [ ]:
# Plot clean cube
print("\n>>> Plotting Clean Cube <<<\n")
fig_clean = create_surface_plot(clean)
fig_clean.show()

In [ ]:
# Plot cube with removed face
print("\n>>> Plotting Cube with Removed Face <<<\n")
fig_handled = create_surface_plot(handled)
fig_handled.show()

## pySurgery Decision Stage (Before Surgery)

In [ ]:
pre_result = analyze_homeomorphism_2d_result(clean["cc"], handled["cc"])
pre_witness = build_homeomorphism_witness(c1=clean["cc"], c2=handled["cc"], dim=2)

print("Analyzer status:", pre_result.status)
print("Analyzer theorem:", pre_result.theorem)
print("Analyzer reasoning:", pre_result.reasoning)

print("\nWitness status:", pre_witness.status)
if pre_witness.witness is not None:
    print("Witness kind:", pre_witness.witness.kind)
    print("Witness theorem:", pre_witness.witness.theorem)
else:
    print("Missing data:", pre_witness.missing_data)

## Surgery Step: Close the Open Face

For this synthetic demo, we add back the removed bottom face triangles to restore
the complete cube, then recompute homology to verify topology is restored.

In [ ]:
def surgery_restore_cube_face():
    """
    Take the modified cube and add back the two removed triangles to restore the complete cube.
    """
    st = gudhi.SimplexTree()

    # Add all vertices
    for i in range(8):
        st.insert([i])

    # Define ALL 12 triangles (including the restored bottom face)
    triangles = [
        # Bottom face (z=0) - NOW RESTORED
        (0, 1, 2),
        (0, 2, 3),
        # Top face (z=1)
        (4, 6, 5),
        (4, 7, 6),
        # Front face (y=0)
        (0, 5, 1),
        (0, 4, 5),
        # Back face (y=1)
        (3, 2, 6),
        (3, 6, 7),
        # Left face (x=0)
        (0, 3, 7),
        (0, 7, 4),
        # Right face (x=1)
        (1, 5, 6),
        (1, 6, 2),
    ]

    # Add edges (1-skeleton)
    edges_set = set()
    for tri in triangles:
        for i in range(3):
            for j in range(i+1, 3):
                edge = tuple(sorted([tri[i], tri[j]]))
                edges_set.add(edge)

    for edge in edges_set:
        st.insert(list(edge))

    # Add triangles
    for tri in triangles:
        st.insert(list(tri))

    # Use same vertices as original cube
    vertices = np.array([
        [0, 0, 0],
        [1, 0, 0],
        [1, 1, 0],
        [0, 1, 0],
        [0, 0, 1],
        [1, 0, 1],
        [1, 1, 1],
        [0, 1, 1],
    ], dtype=float)

    return st, vertices


post_st, post_pts = surgery_restore_cube_face()
print("post-surgery simplex tree:", post_st.num_vertices(), "vertices,", post_st.num_simplices(), "simplices")

In [ ]:
post = build_surface_record_from_simplex_tree(post_st, post_pts, "post_surgery_cube")

print_surface_summary(post)

post_result = analyze_homeomorphism_2d_result(clean["cc"], post["cc"])
post_witness = build_homeomorphism_witness(c1=clean["cc"], c2=post["cc"], dim=2)

print("\nPost-surgery analyzer status:", post_result.status)
print("Post-surgery reasoning:", post_result.reasoning)

print("\nPost-surgery witness status:", post_witness.status)
if post_witness.witness is not None:
    print("Post-surgery witness kind:", post_witness.witness.kind)
    print("Post-surgery theorem:", post_witness.witness.theorem)
else:
    print("Post-surgery missing data:", post_witness.missing_data)

In [ ]:
# Plot before surgery (cube with removed face)
print("\n>>> Plotting Before Surgery (Incomplete Cube) <<<\n")
fig_before = create_surface_plot(handled)
fig_before.update_layout(
    title="Before Surgery: Cube with Removed Bottom Face",
)
fig_before.show()

# Plot after surgery (restored cube)
print("\n>>> Plotting After Surgery (Restored Cube) <<<\n")
fig_after = create_surface_plot(post)
fig_after.update_layout(
    title="After Surgery: Complete Restored Cube",
)
fig_after.show()

In [ ]:
def short_h(rec):
    return {
        "H0_rank": rec["homology"]["H0"][0],
        "H1_rank": rec["homology"]["H1"][0],
        "H2_rank": rec["homology"]["H2"][0],
        "chi": rec["counts"]["chi"],
    }

print("Clean cube:", short_h(clean))
print("Cube with removed face (before surgery):", short_h(handled))
print("Post-surgery restored cube:", short_h(post))

print("\nDecision summary:")
print("Before surgery ->", pre_result.status)
print("After surgery  ->", post_result.status)

